In [8]:
from pathlib import Path
from IPython.display import display

import matplotlib.pyplot as plt
import pandas as pd

RESULTS_DIR = Path(".")
COLS = ["run_name", "epoch", "category", "n_examples", "accuracy_mean", "accuracy_std"]
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)


def load_accuracy_by_category(name, csv_path, exclude_categories=None):
    if not csv_path.exists():
        print(f"skip {name}: missing {csv_path.name}")
        return None

    df = pd.read_csv(csv_path)
    df = df[(df["testset"].isin(["dative_accusative_pairs", "dative_accusative_pairs_shifted"])) & (df["n_seeds"].isin([1, 4, 5]))]
    df = df[COLS].copy()
    if df.empty:
        print(f"skip {name}: no matching rows")
        return None

    df = df.sort_values(["category", "epoch"])
    df["accuracy_mean"] = (df["accuracy_mean"] * 100).round(2)
    df["accuracy_std"] = (df["accuracy_std"] * 100).round(2)
    df["run_name"] = name

    if exclude_categories:
        df = df[~df["category"].isin(exclude_categories)]
        if df.empty:
            print(f"skip {name}: no categories left after exclude_categories={exclude_categories}")
            return None
    return df


def display_tables_by_epoch(frames, epochs=range(-1,6)):
    if not frames:
        print("no model tables to display")
        return
    all_df = pd.concat(frames, ignore_index=True)
    for epoch in epochs:
        sub = all_df[all_df["epoch"] == epoch]
        if sub.empty:
            continue
        table = (
            sub.pivot(index="run_name", columns="category", values="accuracy_mean")
            .sort_index()
            .reset_index()
            .rename(columns={"run_name": "model"})
        )
        table.columns.name = None
        print(f"\n## epoch {epoch}")
        display(
            table.style.set_properties(
                subset=["model"],
                **{"white-space": "nowrap", "text-align": "left"},
            )
        )


def plot_accuracy_by_category(name, csv_path, out_path=None, exclude_categories=None):
    plot_df = load_accuracy_by_category(name, csv_path, exclude_categories=exclude_categories)
    if plot_df is None:
        return None

    y_min = plot_df["accuracy_mean"].min() - 3

    fig, ax = plt.subplots(figsize=(8, 5))
    for category, group in plot_df.groupby("category", sort=True):
        ax.plot(group["epoch"], group["accuracy_mean"], marker="o", label=str(category))
        ax.fill_between(
            group["epoch"],
            group["accuracy_mean"] - group["accuracy_std"],
            group["accuracy_mean"] + group["accuracy_std"],
            alpha=0.2,
        )

    ax.set_xlabel("epoch")
    ax.set_ylabel("accuracy (%)")
    ax.set_ylim(bottom=y_min)
    ax.set_title(f"{name} — dative_accusative_pairs")
    ax.legend(title="category", bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    if out_path is None:
        out_path = csv_path.parent / f"{name}_accuracy.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return plot_df


In [ ]:
for folder in sorted(p for p in RESULTS_DIR.iterdir() if p.is_dir()):
    name = folder.name
    csv_path = folder / f"{name}_aggregate_by_category.csv"
    out_path = f"{name}_accuracy.png"
    # print(f"plotting {name} at {out_path}")
    plot_accuracy_by_category(name, csv_path, out_path)

In [9]:
frames = []
for folder in sorted(p for p in RESULTS_DIR.iterdir() if p.is_dir()):
    name = folder.name
    csv_path = folder / f"{name}_aggregate_by_category.csv"
    out_path = f"{name}_accuracy.png"
    df = plot_accuracy_by_category(name, csv_path, out_path)
    if df is not None:
        frames.append(df)
display_tables_by_epoch(frames)

skip morph-char-en-de_phase_2_regular_de_babylm_shifted: missing morph-char-en-de_phase_2_regular_de_babylm_shifted_aggregate_by_category.csv

## epoch 1


,model,ALL,Ablative_TR,Instrumental_TR,Locative_TR,Mixed_TR,NaN_prep,dative-case,no-case,other-case
0,morph-char-de_phase_2_de_babylm_regular,77.260000,86.840000,96.390000,73.150000,88.310000,61.260000,33.150000,45.070000,36.070000
1,morph-char-de_phase_2_de_babylm_shifted,76.250000,87.120000,95.700000,72.230000,86.240000,61.080000,31.020000,42.390000,34.320000
2,morph-char-de_phase_2_de_nllb_regular,79.380000,87.820000,96.140000,78.010000,88.320000,64.370000,25.510000,33.280000,29.070000
3,morph-char-de_phase_2_de_nllb_shifted,79.520000,88.610000,96.430000,77.920000,87.970000,65.030000,27.090000,35.520000,30.270000
4,morph-char-en-de_phase_2_regular_de_babylm_regular,80.960000,90.510000,97.710000,76.730000,93.440000,70.540000,32.440000,55.070000,31.910000
5,morph-char-en-de_phase_2_regular_de_nllb_regular,45.530000,51.960000,49.920000,42.650000,43.360000,57.840000,45.590000,46.420000,54.320000
6,morph-char-en-de_phase_2_regular_de_nllb_reset_embeddings,45.310000,41.830000,41.120000,46.910000,44.530000,42.810000,51.100000,58.210000,41.640000
7,morph-char-en-de_phase_2_regular_de_nllb_reset_embeddings_shifted,48.000000,45.550000,46.700000,49.700000,46.520000,46.170000,45.590000,48.810000,46.890000
8,morph-char-en-de_phase_2_regular_de_nllb_shifted,51.630000,47.360000,48.110000,50.240000,62.430000,42.510000,49.920000,61.940000,38.470000
9,morph-char-tr-de_phase_2_regular_de_nllb_regular,46.930000,50.590000,50.830000,47.440000,43.550000,55.870000,31.260000,22.840000,47.320000



## epoch 3


,model,ALL,Ablative_TR,Instrumental_TR,Locative_TR,Mixed_TR,NaN_prep,dative-case,no-case,other-case
0,morph-char-de_phase_2_de_babylm_regular,82.080000,91.120000,97.740000,78.280000,94.900000,69.400000,34.650000,54.330000,31.260000
1,morph-char-de_phase_2_de_babylm_shifted,82.000000,90.850000,97.690000,78.360000,94.430000,69.880000,32.050000,57.010000,32.240000
2,morph-char-de_phase_2_de_nllb_regular,83.570000,91.320000,97.970000,81.360000,94.530000,72.810000,30.470000,48.060000,34.430000
3,morph-char-de_phase_2_de_nllb_shifted,83.460000,91.150000,97.900000,80.930000,94.660000,73.230000,32.280000,48.810000,35.850000
4,morph-char-en-de_phase_2_regular_de_babylm_regular,84.620000,93.400000,98.420000,80.450000,96.210000,78.140000,41.650000,65.670000,41.530000
5,morph-char-en-de_phase_2_regular_de_nllb_regular,40.790000,48.190000,47.610000,36.720000,42.250000,48.260000,34.570000,38.660000,46.780000
6,morph-char-en-de_phase_2_regular_de_nllb_reset_embeddings,44.180000,38.120000,39.020000,46.260000,43.690000,43.230000,50.870000,59.400000,41.090000
7,morph-char-en-de_phase_2_regular_de_nllb_reset_embeddings_shifted,44.330000,43.470000,44.140000,45.510000,42.750000,43.050000,41.500000,43.430000,43.280000
8,morph-char-en-de_phase_2_regular_de_nllb_shifted,59.050000,59.480000,62.450000,56.340000,62.570000,57.540000,63.390000,77.460000,58.580000
9,morph-char-tr-de_phase_2_regular_de_nllb_regular,50.410000,55.190000,56.570000,47.740000,51.020000,61.200000,41.890000,34.630000,55.960000



## epoch 5


,model,ALL,Ablative_TR,Instrumental_TR,Locative_TR,Mixed_TR,NaN_prep,dative-case,no-case,other-case
0,morph-char-de_phase_2_de_babylm_regular,83.350000,92.660000,98.060000,79.460000,95.460000,73.230000,36.690000,60.750000,34.860000
1,morph-char-de_phase_2_de_babylm_shifted,83.530000,92.660000,98.170000,79.920000,95.310000,72.460000,35.670000,60.750000,35.960000
2,morph-char-de_phase_2_de_nllb_regular,84.750000,91.960000,98.010000,82.590000,95.600000,74.430000,35.280000,52.390000,36.170000
3,morph-char-de_phase_2_de_nllb_shifted,84.480000,92.340000,97.970000,81.870000,95.720000,74.730000,34.570000,54.630000,36.170000
4,morph-char-en-de_phase_1_regular,46.120000,53.760000,49.190000,43.530000,42.290000,57.900000,50.240000,48.810000,56.170000
5,morph-char-en-de_phase_1_shuffled,46.810000,51.830000,52.570000,44.070000,46.350000,59.220000,43.390000,31.190000,54.430000
6,morph-char-en-de_phase_2_regular_de_babylm_regular,85.190000,94.030000,98.700000,80.980000,96.200000,79.040000,44.570000,67.460000,44.920000
7,morph-char-en-de_phase_2_regular_de_nllb_regular,38.930000,46.510000,45.490000,34.760000,41.230000,44.790000,33.070000,34.930000,43.280000
8,morph-char-en-de_phase_2_regular_de_nllb_reset_embeddings,43.880000,37.600000,38.830000,46.010000,43.280000,43.290000,50.080000,59.100000,41.310000
9,morph-char-en-de_phase_2_regular_de_nllb_reset_embeddings_shifted,43.610000,43.100000,43.660000,44.640000,42.040000,42.630000,40.470000,41.790000,42.840000
